<a href="https://colab.research.google.com/github/hiiamlars/master_thesis/blob/main/tvd_mi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Setup

In [ ]:
# @title Dependencies

# Installations of third-party libraries
!pip install openai -q
!pip install anthropic -q
!pip install together -q

# First-party imports
import os
import json
import csv
import time
import random
from typing import List, Dict, Any, Optional, Tuple

# Third-party imports
import pandas as pd
from openai import OpenAI
from anthropic import Anthropic
from together import Together
from google.colab import drive

print("\nInstalling and loading dependencies complete!")

In [ ]:
# @title Paths

# Mount GDrive
drive.mount('/content/drive')

# Set working directory
WORKING_DIR = "/content/drive/MyDrive/Thesis/"

# Create working directory (if not already existing)
os.makedirs(WORKING_DIR, exist_ok=True)
print(f"Ensured working directory exists at: {WORKING_DIR}.")

# Define and create PAPER_DIR for paper content
PAPER_DIR = os.path.join(WORKING_DIR, "iclr/final")
os.makedirs(PAPER_DIR, exist_ok=True)
print(f"Ensured raw directory exists at: {PAPER_DIR}.")

# Define and create LLM_DIR for llm based reviews
LLM_DIR = os.path.join(WORKING_DIR, "llm_reviews")
os.makedirs(LLM_DIR, exist_ok=True)
print(f"Ensured llm review directory exists at: {LLM_DIR}.")

# Define and create RES_DIR for the result file
RES_DIR = os.path.join(WORKING_DIR, "tvd_mi")
os.makedirs(RES_DIR, exist_ok=True)
print(f"Ensured result directory exists at: {RES_DIR}.")

# Define and create RES_DIR for the log file
LOG_DIR = os.path.join(WORKING_DIR, "tvd_mi")
os.makedirs(LOG_DIR, exist_ok=True)
print(f"Ensured logging directory exists at: {LOG_DIR}.")

In [ ]:
# @title Definitions

# Active model
MODEL = "gpt-4o-mini"
# MODEL = "claude-3-5-haiku-20241022"
# MODEL = "meta-llama/Llama-3.3-70B-Instruct-Turbo"

# Depay for retry
TIMEOUT_SECONDS = 1200

# API Keys
OPENAI_KEY = "OPENAI_KEY_PLACEHOLDER"
ANTHROPIC_KEY = "ANTHROPIC_KEY_PLACEHOLDER"
TOGETHER_KEY = "TOGETHER_KEY_PLACEHOLDER"

# Define the type of TVD-MI score to calculate using a numerical option
# 1: "paper / human review"
# 2: "paper / llm review"
# 3: "human review / llm review"
# 4: "llm review / llm review"
TVD_MI_SCORE_OPTION = 2

# Define col names for the result file for combined TPR and FPR on one row
RES_COL = [
    'paper_id',
    'tpr_response_a', 'tpr_response_b', 'tpr_tvd_mi_score_1', 'tpr_tvd_mi_score_2', 'tpr_avg_tvd_mi_score',
    'fpr_response_a', 'fpr_response_b', 'fpr_tvd_mi_score_1', 'fpr_tvd_mi_score_2', 'fpr_avg_tvd_mi_score'
]

# Define parameter columns for grouping and logging
PARAM_COLS = ["model", "temperature", "reasoning_effort", "chain_of_thought", "scope"]

# Define log file columns, including parameter configurations
LOG_COL = ['paper_id'] + PARAM_COLS + ['TPR_calculated', 'FPR_calculated', 'complete']

SIMULATION_ACTIVE = True # SIMULATION_ACTIVE = False activates the API calls

print("\nDefintions complete!")

In [ ]:
# @title Load data

# Read dataset_paper
dataset_paper = pd.read_parquet(os.path.join(PAPER_DIR, "dataset_paper.parquet"))

# Select 'paper_id' and the target columns for paper content and human reviews
paper_content = dataset_paper[['paper_id', 'parsed_text', 'human_review1', 'human_review2', 'human_review3']] # change to human_review_1 etc.

# Check paper_content
print(paper_content.head(3))

# Read llm reviews
dataset_llm = pd.read_csv(os.path.join(LLM_DIR, "llm_sim_results.csv"))

# Check dataset_llm
print(dataset_llm.head(3))

print("\nData loaded!")

In [ ]:
# @title Merge data

# Merge paper content with LLM data
merged_df = pd.merge(paper_content, dataset_llm, on='paper_id', how='outer')
print(merged_df.head(3))
print(merged_df.shape)

print("\nData merged!")

# API

These functions are responsible for generating the prompt sent to the LLM for TVD-MI evaluation and then interpreting the LLM's response into a numerical score. They are core to how the TVD-MI methodology interacts with the language model.

In [7]:
def generate_tvd_mi_prompt(response_a: str, response_b: str) -> str:

    """Generate prompt for TVD-MI critic"""

    prompt = f"""You are evaluating whether two responses could be from agents working on the same task.

Task description: Scientific paper peer review task

Response A: {response_a}

Response B: {response_b}

Do these responses show evidence of coming from the same task/source? Consider:
- Shared specific details, facts, or entities
- Similar topics or themes
- Overlapping information that would be unlikely if from different sources

Respond in two sentences or less. Put the information gain at the end in brackets in this format:

Rate the information gain:
- [[Significant Gain]]: Clear evidence they're from the same source (many shared specifics)
- [[Little Gain]]: Some shared elements but also differences
- [[No Gain]]: No evidence of shared source (could be from completely different tasks)""" # Should I adapt the first sentence for my own purpose?

    return prompt

def interpret_tvd_mi_response(response: str) -> float:

    """Convert LLM response to numeric score"""

    response = response.strip().lower()

    if "[significant gain]" in response:
        return 1.0
    elif "[little gain]" in response:
        return 0.25
    elif "[no gain]" in response:
        return 0.0
    elif "[total failure]" in response:
        return "TOTAL FAILURE"
    else:
        # Default to no gain if response is unclear
        print(f"Warning: Unclear response '{response}', defaulting to [[No Gain]]")
        return 0.0

This class provides a unified interface for interacting with different LLM APIs (OpenAI, Anthropic, Together) and supports a simulation mode for testing without making actual API calls. It handles API key management and retry logic.

In [8]:
class LLMClient:
    """
    Unified client handling Simulation, Caching, and Routing
    for OpenAI, Anthropic, and Together.
    """

    def __init__(self, simulate: bool = True, seed: int = 7):
        self.simulate = simulate
        self.rng = random.Random(seed)
        self.cache = {}
        self._openai = None
        self._anthropic = None
        self._together = None

    def _maybe_init_clients(self):
        """
        Initialize clients if simulation is off
        """
        if self.simulate:
            return

        if self._openai is None and "OPENAI_KEY" in globals():
            self._openai = OpenAI(api_key=OPENAI_KEY)
        if self._anthropic is None and "ANTHROPIC_KEY" in globals():
            self._anthropic = Anthropic(api_key=ANTHROPIC_KEY)
        if self._together is None and "TOGETHER_KEY" in globals():
            self._together = Together(api_key=TOGETHER_KEY)

    def call(
        self,
        model: str,
        prompt: str,
        temperature: float = 0.0,
        max_tokens: int = 500,
        max_retries: int = 5
    ) -> Tuple[str, Dict[str, Any]]:
        """
        Call the API / simulate with the given parameters
        """

        self._maybe_init_clients()

        # Simulation
        if self.simulate:

            # Create artificial response
            raw = {
                "model": model,
                "temperature": temperature,
                "max_tokens": max_tokens,
                "provider": "simulated",
                "response": (
                    f"[significant gain]"
                )
            }

            # Make the response human readable
            structured = json.dumps(raw, indent=2)

            return structured, raw

        # APIs
        for attempt in range(max_retries):
            try:

                # Create result items
                response_text = ""
                provider = ""

                # ANTHROPIC
                if "claude" in model.lower():
                    resp = self._anthropic.messages.create(
                        model=model,
                        max_tokens=max_tokens,
                        temperature=temperature,
                        messages=[{"role": "user", "content": prompt}]
                    )
                    response_text = resp.content[0].text
                    provider = "anthropic"

                # OPENAI
                elif "gpt" in model.lower():
                    resp = self._openai.chat.completions.create(
                        model=model,
                        messages=[{"role": "user", "content": prompt}],
                        temperature=temperature,
                        max_tokens=max_tokens
                    )
                    response_text = resp.choices[0].message.content
                    provider = "openai"

                # TOGETHER
                else:
                    resp = self._together.chat.completions.create(
                        model=model,
                        messages=[{"role": "user", "content": prompt}],
                        temperature=temperature,
                        max_tokens=max_tokens
                    )
                    response_text = resp.choices[0].message.content
                    provider = "together"

                # Create full response
                raw = {
                    "model": model,
                    "temperature": temperature,
                    "max_tokens": max_tokens,
                    "provider": provider,
                    "response": response_text
                }

                # Make the response human readable
                structured = json.dumps(raw, indent=2)

                return structured, raw

            # Run again after delay until max_retries
            except Exception as e:
                wait_time = (2 ** attempt) + random.random()
                print(f"Error on attempt {attempt+1}: {e}. Retrying in {wait_time:.2f}s...")
                time.sleep(wait_time)

        # Create error in case of failure till max_retries
        error_raw = {"status": "error", "model": model, "response": "[TOTAL FAILURE]"}

        # Make error human readable
        return json.dumps(error_raw, indent=2), error_raw

`configure_tvd_mi_run` sets up the file paths and column selections based on your `TVD_MI_SCORE_OPTION`. `calculate_tvd_mi_scores_for_pair` then orchestrates the LLM calls for a given pair of responses and returns all relevant scores.

In [9]:
def configure_tvd_mi_run(option: int, res_dir: str, log_dir: str) -> Tuple[List[str], str, str]:
    """
    Configures the column pair and output file names based on the TVD-MI score option.
    Randomly selects human/LLM review columns if multiple are available.
    """
    column_pair = []
    file_suffix = ""
    score_type_label = ""

    # Dynamically get all human review columns
    all_human_review_cols = [col for col in paper_content.columns if col.startswith('human_review_')]
    # Dynamically get all LLM review columns
    all_llm_review_cols = [col for col in dataset_llm.columns if col.startswith('llm_review_')]

    # Randomly select one human review column if available
    selected_human_review_col = random.choice(all_human_review_cols) if all_human_review_cols else ""
    # Randomly select one LLM review column if available
    selected_llm_review_col = random.choice(all_llm_review_cols) if all_llm_review_cols else ""

    if option == 1:
        score_type_label = "paper / human review"
        if not selected_human_review_col:
            raise ValueError("No human review columns available for 'paper / human review' comparison.")
        column_pair = ["parsed_text", selected_human_review_col]
        file_suffix = f"paper_{selected_human_review_col}"

    elif option == 2:
        score_type_label = "paper / llm review"
        if not selected_llm_review_col:
            raise ValueError("No LLM review columns available for 'paper / llm review' comparison.")
        column_pair = ["parsed_text", selected_llm_review_col]
        file_suffix = f"paper_{selected_llm_review_col}"

    elif option == 3:
        score_type_label = "human review / llm review"
        if not selected_human_review_col or not selected_llm_review_col:
            raise ValueError("Not enough human/LLM review columns available for 'human review / llm review' comparison.")
        column_pair = [selected_human_review_col, selected_llm_review_col]
        file_suffix = f"h_{selected_human_review_col}_l_{selected_llm_review_col}"

    elif option == 4:
        score_type_label = "llm review / llm review"
        # Check if there are sufficient llm reviews for this case
        if len(all_llm_review_cols) < 2:
            raise ValueError("Not enough LLM review columns available for 'llm review / llm review' comparison (need at least two distinct ones).")

        # Select the first LLM review column
        selected_llm_review_col_1 = selected_llm_review_col

        # Select a second, *different* LLM review column
        remaining_llm_review_cols = [col for col in all_llm_review_cols if col != selected_llm_review_col_1]
        selected_llm_review_col_2 = random.choice(remaining_llm_review_cols)

        column_pair = [selected_llm_review_col_1, selected_llm_review_col_2]
        file_suffix = f"llm_{selected_llm_review_col_1}_llm_{selected_llm_review_col_2}"
    else:
        raise ValueError(f"Invalid TVD_MI_OPTION: {option}. Please choose between 1 and 4.")

    output_path = os.path.join(res_dir, f"tvd_mi_scores_{file_suffix}.csv")
    log_path = os.path.join(log_dir, f"tvd_mi_log_{file_suffix}.csv")

    print(f"\nConfigured for TVD-MI type: '{score_type_label}'")
    print(f"Using columns: {column_pair[0]} and {column_pair[1]}")
    print(f"Results will be saved to: {output_path}")
    print(f"Logs will be saved to: {log_path}")

    return column_pair, output_path, log_path

def calculate_tvd_mi_scores_for_pair(llm_client, model, response_a_text, response_b_text):
    """
    Generates prompts, calls LLM, and interprets scores for a given pair of responses.
    """
    # Generate the TVD-MI prompt for order A, B
    tvd_mi_prompt_1 = generate_tvd_mi_prompt(response_a_text, response_b_text)

    # Call the LLMClient for the first time A, B
    structured_response_1, raw_response_1 = llm_client.call(
        model=model,
        prompt=tvd_mi_prompt_1,
        temperature=0.0,
        max_tokens=500
    )

    # Interpret the LLM's response to get the TVD-MI score A, B
    llm_critic_response_text_1 = raw_response_1.get('response')
    tvd_mi_score_1 = interpret_tvd_mi_response(llm_critic_response_text_1)

    # Generate the TVD-MI prompt for order B, A
    tvd_mi_prompt_2 = generate_tvd_mi_prompt(response_b_text, response_a_text)

    # Call the LLMClient for the second time B, A
    structured_response_2, raw_response_2 = llm_client.call(
        model=model,
        prompt=tvd_mi_prompt_2,
        temperature=0.0,
        max_tokens=500
    )

    # Interpret the LLM's response to get the TVD-MI score B, A
    llm_critic_response_text_2 = raw_response_2.get('response')
    tvd_mi_score_2 = interpret_tvd_mi_response(llm_critic_response_text_2)

    # Calculate the average TVD-MI score or set to 'TOTAL FAILURE'
    if tvd_mi_score_1 == "TOTAL FAILURE" or tvd_mi_score_2 == "TOTAL FAILURE":
        avg_tvd_mi_score = "TOTAL FAILURE"
    else:
        avg_tvd_mi_score = (tvd_mi_score_1 + tvd_mi_score_2) / 2

    return {
        'response_a': response_a_text,
        'response_b': response_b_text,
        'prompt_1': tvd_mi_prompt_1,
        'raw_response_1': json.dumps(raw_response_1),
        'structured_response_1': structured_response_1,
        'prompt_2': tvd_mi_prompt_2,
        'raw_response_2': json.dumps(raw_response_2),
        'structured_response_2': structured_response_2,
        'tvd_mi_score_1': tvd_mi_score_1,
        'tvd_mi_score_2': tvd_mi_score_2,
        'avg_tvd_mi_score': avg_tvd_mi_score
    }

print("\nFunctions defined!")


Functions defined!


The `process_single_paper` is the function which sets the `response`s for the prompt, calls the client-related functions and stores the returned and already converted scores and log-information.

In [12]:
def process_single_paper(llm_client, model, paper_id, row, response_a_col_name, response_b_col_name, merged_df, writer_results, writer_log, param_cols):
    """
    Processes a single paper_id, calculating TPR and FPR scores and writing them to result and log files.
    Returns a list of log messages generated during processing.
    """
    messages = []

    messages.append(f"  Processing paper_id: {paper_id}")

    # --- TPR Calculation: response_a and response_b from the current row ---
    messages.append(f"    Calculating TPR for paper_id: {paper_id}")
    response_a_tpr_text = str(row[response_a_col_name]) if pd.notna(row[response_a_col_name]) else ""
    response_b_tpr_text = str(row[response_b_col_name]) if pd.notna(row[response_b_col_name]) else ""

    tpr_results_raw = calculate_tvd_mi_scores_for_pair(llm_client, model, response_a_tpr_text, response_b_tpr_text)
    messages.append(f"    LLM calls for TPR completed for paper_id: {paper_id}")

    # Determine if TPR calculation was successful
    tpr_calculated = (tpr_results_raw['avg_tvd_mi_score'] != "TOTAL FAILURE")

    # --- FPR Calculation: response_b from current row, response_a from random different row ---
    messages.append(f"    Calculating FPR for paper_id: {paper_id}")
    # Determine response_b for FPR from the current row's response_b_col_name
    response_b_fpr_text = str(row[response_b_col_name]) if pd.notna(row[response_b_col_name]) else ""

    # Get all possible indices from the full merged_df, excluding the current row's index 'i'
    # (Using merged_df.index.drop(row.name) to get indices excluding the current row's actual index)
    eligible_indices_for_a = merged_df.index.drop(row.name).tolist()

    if not eligible_indices_for_a:
        response_a_fpr_text = ""
        messages.append(f"    Warning: No other rows available to pick random response_a for FPR for paper_id {paper_id}. Assigning empty string.")
    else:
        random_a_idx = random.choice(eligible_indices_for_a)
        response_a_fpr_text = str(merged_df.loc[random_a_idx, response_a_col_name]) if pd.notna(merged_df.loc[random_a_idx, response_a_col_name]) else ""

    fpr_results_raw = calculate_tvd_mi_scores_for_pair(llm_client, model, response_a_fpr_text, response_b_fpr_text)
    messages.append(f"    LLM calls for FPR completed for paper_id: {paper_id}")

    # Determine if FPR calculation was successful
    fpr_calculated = (fpr_results_raw['avg_tvd_mi_score'] != "TOTAL FAILURE")

    # --- Combine TPR and FPR results for single row in result CSV ---
    data_to_write = {'paper_id': paper_id}

    # Add TPR results with 'tpr_' prefix, extracting only relevant fields for main CSV
    data_to_write['tpr_response_a'] = tpr_results_raw['response_a']
    data_to_write['tpr_response_b'] = tpr_results_raw['response_b']
    data_to_write['tpr_tvd_mi_score_1'] = tpr_results_raw['tvd_mi_score_1']
    data_to_write['tpr_tvd_mi_score_2'] = tpr_results_raw['tvd_mi_score_2']
    data_to_write['tpr_avg_tvd_mi_score'] = tpr_results_raw['avg_tvd_mi_score']

    # Add FPR results with 'fpr_' prefix, extracting only relevant fields for main CSV
    data_to_write['fpr_response_a'] = fpr_results_raw['response_a']
    data_to_write['fpr_response_b'] = fpr_results_raw['response_b']
    data_to_write['fpr_tvd_mi_score_1'] = fpr_results_raw['tvd_mi_score_1']
    data_to_write['fpr_tvd_mi_score_2'] = fpr_results_raw['tvd_mi_score_2']
    data_to_write['fpr_avg_tvd_mi_score'] = fpr_results_raw['avg_tvd_mi_score']

    # Write the combined data to the main results file
    try:
        writer_results.writerow(data_to_write)
        messages.append(f"    Successfully wrote combined results for paper_id: {paper_id}")
    except Exception as e:
        messages.append(f"    FAILED to write combined results for paper_id: {paper_id}. Error: {e}")

    # --- Write combined log data to the log file ---
    log_data_to_write = {
        'paper_id': paper_id,
        'TPR_calculated': tpr_calculated,
        'FPR_calculated': fpr_calculated,
        'complete': tpr_calculated and fpr_calculated
    }
    # Add parameter values to the log data
    for param_col in param_cols:
        log_data_to_write[param_col] = row[param_col]

    try:
        writer_log.writerow(log_data_to_write)
        messages.append(f"    Successfully wrote log for paper_id: {paper_id}")
    except Exception as e:
        messages.append(f"    FAILED to write log for paper_id: {paper_id}. Error: {e}")

    messages.append(f"  Finished processing both TPR and FPR for paper_id: {paper_id}\n")
    return messages

print("\n Function defined!")


 Function defined!


### TVD-MI score calculation

In [14]:
# Helper to handle NaN values when creating hashable keys for comparison
def _nan_to_none(x):
    try:
        if pd.isna(x):
            return None
    except Exception:
        pass
    return x

# Function to create a unique hashable key for a paper-parameter combination
def get_unique_combo_key(row, param_cols):
    key_elements = [row['paper_id']]
    for col in param_cols:
        key_elements.append(_nan_to_none(row[col]))
    return tuple(key_elements)

# Activate/Deactivate simulation/API calls
llm_client = LLMClient(simulate=SIMULATION_ACTIVE)

# Configure the run based on the TVD_MI_SCORE_OPTION
column_pair, result_path, log_path = configure_tvd_mi_run(TVD_MI_SCORE_OPTION, RES_DIR, LOG_DIR)

# Select prompt input based on column_pair
response_a_col_name = column_pair[0]
response_b_col_name = column_pair[1]

# Information output
print(f"Prompt input will use columns {response_a_col_name} and {response_b_col_name}", flush = True)

# Check if result file exists to determine if header needs to be written
results_file_exists = os.path.exists(result_path)

# Check if log file exists to determine if header needs to be written
log_file_exists = os.path.exists(log_path)

# Information output
print(f"Processing and saving results to: {result_path}", flush = True)
print(f"Logging successful iterations to: {log_path}", flush = True)

# Build the "done" set from existing log file
done_combinations = set()
if os.path.exists(log_path):
    try:
        log_df = pd.read_csv(log_path)
        # Ensure 'complete' column is boolean for filtering
        if 'complete' in log_df.columns:
            completed_log_df = log_df[log_df['complete'] == True]
            for _, log_row in completed_log_df.iterrows():
                done_combinations.add(get_unique_combo_key(log_row, PARAM_COLS))
            print(f"Loaded {len(done_combinations)} previously completed paper/parameter combinations from {log_path}. These will be skipped.", flush=True)
        else:
            print(f"Warning: 'complete' column not found in log file {log_path}. Cannot determine previously completed combinations.", flush=True)
    except Exception as e:
        print(f"Warning: Could not load or parse existing log file {log_path}. Error: {e}. Starting fresh for all combinations.", flush=True)


# Open both files in append mode
with open(result_path, 'a', newline='', encoding='utf-8') as f_results, \
     open(log_path, 'a', newline='', encoding='utf-8') as f_log:

    writer_results = csv.DictWriter(f_results, fieldnames=RES_COL)
    writer_log = csv.DictWriter(f_log, fieldnames=LOG_COL)

    # Write headers only if files didn't exist
    if not results_file_exists:
        writer_results.writeheader()

    if not log_file_exists:
        writer_log.writeheader()

    # Get total number of rows for progress tracking
    total_rows = len(merged_df)

    # Group by parameter columns and then iterate through each group
    # Using sorted() to ensure consistent order of groups across runs
    grouped_df = merged_df.groupby(PARAM_COLS, dropna=False)

    print(f"Starting processing for {len(grouped_df)} parameter configuration groups.", flush=True)

    # Iterate through each parameter configuration group
    for group_idx, (params, group_df) in enumerate(list(grouped_df)[:2]): #currently in test mode with head (2)

        param_values_dict = dict(zip(PARAM_COLS, params))
        print(f"\nProcessing group {group_idx + 1}/{len(grouped_df)} with parameters: {param_values_dict}", flush=True)

        # Iterate through each row (paper_id) within the current parameter group
        for i, row in group_df.head(2).iterrows(): #currently in test mode with head (2)
            paper_id = row['paper_id'] # ID of the paper

            current_combo_key = get_unique_combo_key(row, PARAM_COLS)

            if current_combo_key in done_combinations:
                print(f"  Skipping paper_id: {paper_id} (already complete for parameters: {param_values_dict})", flush=True)
                continue # Skip to the next paper_id in this group

            # Call the new function to process each paper_id and get messages
            processing_messages = process_single_paper(
                llm_client, MODEL, row['paper_id'], row, response_a_col_name,
                response_b_col_name, merged_df, writer_results, writer_log, PARAM_COLS
            )
            # Print messages returned from the function
            for msg in processing_messages:
                print(msg, flush=True)

print("\nProcessing complete!")


Configured for TVD-MI type: 'paper / llm review'
Using columns: parsed_text and llm_review_2
Results will be saved to: /content/drive/MyDrive/Thesis/tvd_mi/tvd_mi_scores_paper_llm_review_2.csv
Logs will be saved to: /content/drive/MyDrive/Thesis/tvd_mi/tvd_mi_log_paper_llm_review_2.csv
Prompt input will use columns parsed_text and llm_review_2
Processing and saving results to: /content/drive/MyDrive/Thesis/tvd_mi/tvd_mi_scores_paper_llm_review_2.csv
Logging successful iterations to: /content/drive/MyDrive/Thesis/tvd_mi/tvd_mi_log_paper_llm_review_2.csv
Starting processing for 2 parameter configuration groups.

Processing group 1/2 with parameters: {'model': 'gpt-5-mini-2025-08-07', 'temperature': np.float64(nan), 'reasoning_effort': 'low', 'chain_of_thought': np.float64(nan), 'scope': 'abstract'}
  Processing paper_id: -0tPmzgXS5
    Calculating TPR for paper_id: -0tPmzgXS5
    LLM calls for TPR completed for paper_id: -0tPmzgXS5
    Calculating FPR for paper_id: -0tPmzgXS5
    LLM ca

In [15]:
# result.csv-file
tvd_mi_results_df = pd.read_csv(result_path)

# Check results
display(tvd_mi_results_df.head(5))

# log.csv-file
tvd_mi_log_df = pd.read_csv(log_path)

# Check logs
display(tvd_mi_log_df.head(5))

print("\nProcessing complete!")

,paper_id,tpr_response_a,tpr_response_b,tpr_tvd_mi_score_1,tpr_tvd_mi_score_2,tpr_avg_tvd_mi_score,fpr_response_a,fpr_response_b,fpr_tvd_mi_score_1,fpr_tvd_mi_score_2,fpr_avg_tvd_mi_score
0,-0tPmzgXS5,INTRODUCTION Video recognition methods has evo...,"{\n ""model"": ""gpt-5-mini-2025-08-07"",\n ""tem...",1.0,1.0,1.0,INTRODUCTION The complexity of high-performanc...,"{\n ""model"": ""gpt-5-mini-2025-08-07"",\n ""tem...",1.0,1.0,1.0
1,-1x2-lp1eZf,INTRODUCTION Spiking neural networks (SNNs) (M...,"{\n ""model"": ""gpt-5-mini-2025-08-07"",\n ""tem...",1.0,1.0,1.0,"INTRODUCTION Text to speech (TTS), or speech s...","{\n ""model"": ""gpt-5-mini-2025-08-07"",\n ""tem...",1.0,1.0,1.0
2,-0tPmzgXS5,INTRODUCTION Video recognition methods has evo...,"{\n ""model"": ""gpt-5-mini-2025-08-07"",\n ""tem...",1.0,1.0,1.0,"INTRODUCTION Temporal networks, which are grap...","{\n ""model"": ""gpt-5-mini-2025-08-07"",\n ""tem...",1.0,1.0,1.0
3,-1x2-lp1eZf,INTRODUCTION Spiking neural networks (SNNs) (M...,"{\n ""model"": ""gpt-5-mini-2025-08-07"",\n ""tem...",1.0,1.0,1.0,INTRODUCTION Bayesian Optimisation (BO) (Shahr...,"{\n ""model"": ""gpt-5-mini-2025-08-07"",\n ""tem...",1.0,1.0,1.0
4,-0tPmzgXS5,INTRODUCTION Video recognition methods has evo...,"{\n ""model"": ""gpt-5-mini-2025-08-07"",\n ""tem...",1.0,1.0,1.0,INTRODUCTION Federated Learning (FL) is an inc...,"{\n ""model"": ""gpt-5-mini-2025-08-07"",\n ""tem...",1.0,1.0,1.0


,paper_id,model,temperature,reasoning_effort,chain_of_thought,scope,TPR_calculated,FPR_calculated,complete
0,-0tPmzgXS5,gpt-5-mini-2025-08-07,NaN,low,NaN,abstract,True,True,True
1,-1x2-lp1eZf,gpt-5-mini-2025-08-07,NaN,low,NaN,abstract,True,True,True
2,-0tPmzgXS5,gpt-5-mini-2025-08-07,NaN,low,NaN,full_text,True,True,True
3,-1x2-lp1eZf,gpt-5-mini-2025-08-07,NaN,low,NaN,full_text,True,True,True
4,-0tPmzgXS5,gpt-5-mini-2025-08-07,NaN,low,NaN,abstract,True,True,True



Processing complete!


# References

The main code logic and in parts exact code has been taken from:

Robertson, Z., & Koyejo, S. (2025). Let's Measure Information Step-by-Step: LLM-Based Evaluation Beyond Vibes. *arXiv preprint arXiv:2508.05469.*

The code can be accessed [here](https://github.com/zrobertson466920/llm-peer-prediction/tree/main).